# 04-5. 동일 summary 그룹 분할

04-4 중복 제거본에서 같은 summary를 공유하는 서로 다른 document가 train·validation·test에 나뉘지 않도록 그룹 단위로 재배치한다. test가 포함된 summary 그룹은 test를 보존하고, 나머지 split의 겹치는 행은 제외한다.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT = Path(r"C:\Users\KDS-30\Desktop\Projects\06_Team_Project4")
assert Path(sys.prefix).resolve() == (PROJECT / ".venv").resolve(), "커널을 Python 3.11 (Team4 News GPU)로 바꾸세요."
INPUT_DIR = PROJECT / "processed_data" / "final_clean_exact_deduplicated"
OUTPUT_DIR = PROJECT / "processed_data" / "final_clean_summary_grouped"
RESULTS = PROJECT / "review_data" / "04-5_summary_group_split"
assert INPUT_DIR.exists(), "먼저 04-4를 실행하세요."
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)
print("입력:", INPUT_DIR)
print("그룹 분할 데이터:", OUTPUT_DIR)
print("분할 기록:", RESULTS)

입력: C:\Users\KDS-30\Desktop\Projects\06_Team_Project4\processed_data\final_clean_exact_deduplicated
그룹 분할 데이터: C:\Users\KDS-30\Desktop\Projects\06_Team_Project4\processed_data\final_clean_summary_grouped
분할 기록: C:\Users\KDS-30\Desktop\Projects\06_Team_Project4\review_data\04-5_summary_group_split


In [2]:
split_order = ["train", "validation", "test"]
frames = [pd.read_csv(INPUT_DIR / f"{split}.csv") for split in split_order]
all_df = pd.concat(frames, ignore_index=True)
assert all_df["row_id"].is_unique
assert all_df["summary"].notna().all()

# summary가 같은 행은 하나의 그룹으로 보고, test를 먼저 고정합니다.
summary_splits = all_df.groupby("summary")["split"].agg(lambda x: set(x))

def assigned_split(source_splits):
    if "test" in source_splits:
        return "test"
    if "validation" in source_splits:
        return "validation"
    return "train"

assignment = summary_splits.map(assigned_split).rename("assigned_split")
all_df = all_df.join(assignment, on="summary")
all_df["summary_group_size"] = all_df.groupby("summary")["row_id"].transform("size")
all_df["summary_group_cross_split"] = all_df.groupby("summary")["split"].transform("nunique").gt(1)

# test에 이미 있는 summary 그룹은 test 행만 남겨 원본 test를 고정합니다.
all_df["keep_after_summary_group"] = ~((all_df["assigned_split"] == "test") & (all_df["split"] != "test"))
all_df["split_action"] = "kept"
all_df.loc[(all_df["assigned_split"] == "test") & (all_df["split"] != "test"), "split_action"] = "excluded_test_summary_overlap"
all_df.loc[all_df["keep_after_summary_group"] & (all_df["split"] != all_df["assigned_split"]), "split_action"] = "moved_to_" + all_df.loc[all_df["keep_after_summary_group"] & (all_df["split"] != all_df["assigned_split"]), "assigned_split"]

manifest = all_df[["row_id", "split", "assigned_split", "split_action", "summary_group_size", "summary_group_cross_split", "title", "link"]].copy()
display(manifest["split_action"].value_counts().rename_axis("action").to_frame("rows"))

,rows
action,
kept,24038
excluded_test_summary_overlap,5
moved_to_validation,2


In [3]:
# assigned_split별로 저장해, 동일 summary가 두 split에 동시에 남지 않게 합니다.
saved = []
for split in split_order:
    output = all_df.loc[all_df["keep_after_summary_group"] & all_df["assigned_split"].eq(split)].copy()
    output.to_csv(OUTPUT_DIR / f"{split}.csv", index=False, encoding="utf-8-sig")
    saved.append(output)

final_df = pd.concat(saved, ignore_index=True)
assert final_df.groupby("summary")["assigned_split"].nunique().max() == 1
assert final_df["split"].eq("test").sum() == all_df["split"].eq("test").sum(), "원본 test 행 수가 바뀌었습니다."

summary = pd.DataFrame({
    "split": split_order,
    "input_rows": [int((all_df["split"] == split).sum()) for split in split_order],
    "output_rows": [int((final_df["assigned_split"] == split).sum()) for split in split_order],
    "excluded_test_summary_overlap": [int(((all_df["split"] == split) & (all_df["split_action"] == "excluded_test_summary_overlap")).sum()) for split in split_order],
    "moved_in": [int(((final_df["assigned_split"] == split) & (final_df["split"] != split)).sum()) for split in split_order],
})
manifest.to_csv(RESULTS / "summary_group_manifest.csv", index=False, encoding="utf-8-sig")
summary.to_csv(RESULTS / "split_summary.csv", index=False, encoding="utf-8-sig")
pd.DataFrame([{
    "input_data": str(INPUT_DIR),
    "output_data": str(OUTPUT_DIR),
    "group_key": "exact summary",
    "assignment_policy": "test fixed; test-overlapping train/validation rows excluded; otherwise validation then train priority",
    "cross_split_summary_groups": int(all_df.groupby("summary")["split"].nunique().gt(1).sum()),
}]).to_csv(RESULTS / "run_info.csv", index=False, encoding="utf-8-sig")
display(summary)
print("최종 행 수:", f"{len(final_df):,}")

,split,input_rows,output_rows,excluded_test_summary_overlap,moved_in
0,train,19242,19236,4,0
1,validation,2239,2240,1,2
2,test,2564,2564,0,0


최종 행 수: 24,040


이후 학습에는 `final_clean_summary_grouped`의 train·validation·test를 사용한다. `summary_group_manifest.csv`에는 각 행의 원래 split, 재배치 split, 제외 사유가 기록된다.